In [1]:
# Import libraries
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
from pathlib import Path
import os
from tqdm.auto import tqdm
from PIL import Image
import warnings
from torchvision import transforms

warnings.filterwarnings("ignore")

# Set random seed
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data Preparation
IMAGE_PATH = Path("/kaggle/input/banana-disease-recognition-dataset/Banana Disease Recognition Dataset/Original Images/Original Images")
IMAGE_PATH_LIST = list(IMAGE_PATH.glob("*/*.jpg"))

# Create dataframe
df = pd.DataFrame({
    'path': [str(img_path) for img_path in IMAGE_PATH_LIST],
    'label': [img_path.parent.stem for img_path in IMAGE_PATH_LIST]
})

# Get classes
classes = sorted(df['label'].unique())
label_map = {cls: i for i, cls in enumerate(classes)}

# Train-Valid-Test Split
df_train, df_rest = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df["label"])
df_valid, df_test = train_test_split(df_rest, test_size=0.5, random_state=SEED, stratify=df_rest["label"])

# Dataset and DataLoader
class CustomImageDataset(Dataset):
    def __init__(self, df, label_map, transform_params):
        self.df = df.reset_index(drop=True)
        self.label_map = label_map
        self.transform = transforms.Compose([
            transforms.Resize(transform_params['resize_size']),
            transforms.CenterCrop(transform_params['crop_size']),
            transforms.ToTensor(),
            transforms.Normalize(mean=transform_params['mean'], std=transform_params['std'])
        ])
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'path']
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        label = self.label_map[self.df.loc[idx, 'label']]
        return image, label

# Define transform parameters
transform_params = {
    'crop_size': 224,
    'resize_size': 256,
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225]
}

# Create datasets and dataloaders
BATCH_SIZE = 8
NUM_WORKERS = os.cpu_count()

train_dataset = CustomImageDataset(df_train, label_map, transform_params)
valid_dataset = CustomImageDataset(df_valid, label_map, transform_params)
test_dataset = CustomImageDataset(df_test, label_map, transform_params)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Model Setup
class BananaDiseaseClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        
        # Freeze layers
        for param in self.vit.conv_proj.parameters():
            param.requires_grad = False
        for param in self.vit.encoder.parameters():
            param.requires_grad = False
            
        # Modify head
        self.vit.heads = nn.Linear(768, num_classes)
        
    def forward(self, x):
        return self.vit(x)

model = BananaDiseaseClassifier(len(classes)).to(device)

# Training Setup
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Functions
def train_epoch(model, dataloader, loss_fn, optimizer):
    model.train()
    total_loss, total_correct = 0, 0
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_correct += (y_pred.argmax(1) == y).sum().item()
        
    return total_loss/len(dataloader), total_correct/len(dataloader.dataset)

def validate(model, dataloader, loss_fn):
    model.eval()
    total_loss, total_correct = 0, 0
    
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            
            total_loss += loss.item()
            total_correct += (y_pred.argmax(1) == y).sum().item()
            
    return total_loss/len(dataloader), total_correct/len(dataloader.dataset)

# Training Loop
EPOCHS = 100
best_valid_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_dataloader, loss_fn, optimizer)
    valid_loss, valid_acc = validate(model, valid_dataloader, loss_fn)
    
    print(f"Epoch: {epoch+1:03} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {valid_loss:.4f} | Val Acc: {valid_acc*100:.2f}%")
    
    # Save best model
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': valid_loss,
            'accuracy': valid_acc,
            'class_to_idx': label_map,
            'transform_params': transform_params,
            'model_config': {
                'num_classes': len(classes),
                'model_type': 'vit_b_16'
            }
        }, 'best_model.pth')

# Evaluation
def evaluate(model, test_dl, label_map):
    model.eval()
    y_true, y_pred = [], []
    
    with torch.inference_mode():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            y_preds = torch.argmax(y_logits, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(y_preds.cpu().numpy())
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=label_map.keys()))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f"\nTest Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Load best model
def load_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path)
    
    model = BananaDiseaseClassifier(checkpoint['model_config']['num_classes'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    return model, checkpoint

best_model, checkpoint = load_model('best_model.pth', device)
evaluate(best_model, test_dataloader, label_map)

# Save complete model
torch.save({
    'model_state_dict': best_model.state_dict(),
    'class_to_idx': label_map,
    'transform_params': transform_params,
    'model_config': checkpoint['model_config']
}, 'complete_model.pth')

print("\nModel saved as 'complete_model.pth'")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:01<00:00, 204MB/s]


Epoch: 001 | Train Loss: 1.1765 | Train Acc: 60.00% | Val Loss: 0.6455 | Val Acc: 86.89%
Epoch: 002 | Train Loss: 0.4387 | Train Acc: 88.42% | Val Loss: 0.4221 | Val Acc: 88.52%
Epoch: 003 | Train Loss: 0.2599 | Train Acc: 96.49% | Val Loss: 0.3367 | Val Acc: 93.44%
Epoch: 004 | Train Loss: 0.1815 | Train Acc: 98.25% | Val Loss: 0.3023 | Val Acc: 93.44%
Epoch: 005 | Train Loss: 0.1367 | Train Acc: 98.60% | Val Loss: 0.2698 | Val Acc: 95.08%
Epoch: 006 | Train Loss: 0.1014 | Train Acc: 99.30% | Val Loss: 0.2458 | Val Acc: 96.72%
Epoch: 007 | Train Loss: 0.0829 | Train Acc: 99.65% | Val Loss: 0.2387 | Val Acc: 95.08%
Epoch: 008 | Train Loss: 0.0656 | Train Acc: 99.65% | Val Loss: 0.2204 | Val Acc: 96.72%
Epoch: 009 | Train Loss: 0.0561 | Train Acc: 99.65% | Val Loss: 0.2135 | Val Acc: 96.72%
Epoch: 010 | Train Loss: 0.0476 | Train Acc: 99.65% | Val Loss: 0.2037 | Val Acc: 96.72%
Epoch: 011 | Train Loss: 0.0412 | Train Acc: 100.00% | Val Loss: 0.1980 | Val Acc: 96.72%
Epoch: 012 | Train L